## Standard DAgger (3 rounds, parallel collection)

For each trained stiffness E in {5e6, 7.5e6, 1e7, 2e7}:
  - Roll out the student in 8 parallel envs (SubprocVecEnv)
  - Label visited states with the **matching** expert
  - Aggregate, retrain student

beta_schedule = [0.1, 0.0, 0.0]: round 1 mixes 10% expert / 90%
student rollouts (warm-start); subsequent rounds are pure student
(true on-policy).

> Ross et al., "A Reduction of Imitation Learning and Structured
> Prediction to No-Regret Online Learning", AISTATS 2011.

In [ ]:
import os
import json
import shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR
from pathlib import Path
from tqdm import tqdm
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv, VecNormalize

from rod_tracking_env import RodTrackingEnv
from training_student import StudentPolicy

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [ ]:
# CONFIGURATION

ORIGINAL_DATASET = "distillation_data/distillation_dataset.npz"
INITIAL_STUDENT  = "results_student_v4_noise/student_policy.pth"
OUTPUT_DIR = Path("results_dagger_fast")

EXPERTS = {
    5e6: {
        "model":   "results_expert_E5e6_v27/best_model.zip",
        "vecnorm": "results_expert_E5e6_v27/vecnorm_best.pkl",
    },
    7.5e6: {
        "model":   "results_expert_E7.5e6_v27/best_model.zip",
        "vecnorm": "results_expert_E7.5e6_v27/vecnorm_best.pkl",
    },
    1e7: {
        "model":   "results_expert_E1e7_v27/best_model.zip",
        "vecnorm": "results_expert_E1e7_v27/vecnorm_best.pkl",
    },
    2e7: {
        "model":   "results_expert_E2e7_v27/best_model.zip",
        "vecnorm": "results_expert_E2e7_v27/vecnorm_best.pkl",
    },
}

ENV_BASE = {
    'n_elem': 20,
    'sim_dt': 2.0e-4,
    'num_steps_per_update': 7,
    'base_length': 1.0,
    'base_radius': 0.05,
    'density': 1000.0,
    'NU': 11.0,
    'n_control_points': 6,
    'alpha': 75.0,
    'max_rate_of_change_of_activation': np.inf,
    'target_v_max': 0.50,
    'boundary': (-0.35, 0.35, 0.90, 1.0, -0.35, 0.35),
    'final_time': 10.0,
    'success_threshold': 0.01,
    'w_dist': 2.0,
    'w_precision': 5.0,
    'w_progress': 1.0,
    'w_smoothness': 0.03,
    'sigma_mult': 1.5,
    'sigma_floor': 0.01,
}

E_MIN, E_MAX = 5e6, 2e7
E_VALUES = [5e6, 7.5e6, 1e7, 2e7]

MAX_DATASET_SIZE = 6_000_000

OBS_NOISE_STD     = 0.002       # noise on the 44 obs features
E_NORM_NOISE_STD = 0.05     # noise on the E_norm conditioning input
GRAD_CLIP         = 1.0     # max gradient norm

# DAgger params
N_DAGGER_ROUNDS = 3
EPISODES_DYN    = 40
EPISODES_STAT   = 20
BETA_SCHEDULE   = [0.1, 0.0, 0.0]
N_PARALLEL_ENVS = 8


# Training params
EPOCHS_PER_ROUND  = 40
BATCH_SIZE        = 1024
LEARNING_RATE     = 2e-4
EARLY_STOP_PAT    = 10

NUM_WORKERS       = 4


In [1]:
# UTILITIES

def normalize_E(E):
    """Log-scale normalize E to [0, 1] — must match training."""
    return (np.log10(E) - np.log10(E_MIN)) / (np.log10(E_MAX) - np.log10(E_MIN))


class ExpertWrapper:
    """Wraps one SAC expert with batch logit prediction.

    Loads the SAC model and its VecNormalize stats (frozen), exposes
    `get_actions_and_logits_batch(obs_batch)` which returns both the
    deterministic post-tanh action (for env stepping if needed) and
    the pre-tanh logit (clamped to +/-5) which is the training target
    for the student.

    Why the deterministic mean instead of a SAC sample
    --------------------------------------------------
    At training time SAC samples actions stochastically. For
    distillation we want repeatable, clean labels, so we use the
    actor's mean logit directly (no sampling noise). Equivalent in
    spirit to `predict(deterministic=True)` but exposes the pre-tanh
    `mean_logit` which `predict(...)` would drop.
    """

    def __init__(self, model_path, vecnorm_path, young_modulus):
        self.E = young_modulus
        self.model = SAC.load(model_path, device="cuda")

        env_params = {**ENV_BASE, 'young_modulus': young_modulus, 'p_static': 0.0}
        dummy_env = DummyVecEnv([lambda: RodTrackingEnv(**env_params)])
        self.vec_norm = VecNormalize.load(vecnorm_path, dummy_env)
        self.vec_norm.training = False
        self.vec_norm.norm_reward = False

    def get_actions_and_logits_batch(self, obs_batch):
        """Vectorized expert prediction.

        Parameters
        ----------
        obs_batch : (N, obs_dim) float32 — raw env observations (44 dims).

        Returns
        -------
        actions : (N, action_dim) — deterministic post-tanh actions
        logits  : (N, action_dim) — pre-tanh logits clamped to +/-5
                  (cap on the regression target; tanh(5) ~= 0.9999
                  so actions are unaffected)
        """
        norm_obs = self.vec_norm.normalize_obs(obs_batch)
        obs_tensor = self.model.policy.obs_to_tensor(norm_obs)[0]

        with torch.no_grad():
            features = self.model.policy.actor.extract_features(
                obs_tensor, self.model.policy.actor.features_extractor
            )
            latent_pi = self.model.policy.actor.latent_pi(features)
            mean_logit = self.model.policy.actor.mu(latent_pi)
            actions = torch.tanh(mean_logit)

        logits = torch.clamp(mean_logit, -5.0, 5.0)
        return actions.cpu().numpy(), logits.cpu().numpy()


def load_student(path, device):
    """Load a StudentPolicy checkpoint into eval mode."""
    ckpt = torch.load(path, map_location=device, weights_only=False)
    obs_dim = ckpt["obs_dim"]
    action_dim = ckpt["action_dim"]
    model = StudentPolicy(obs_dim, action_dim).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    return model, obs_dim, action_dim


def make_env(E, p_static, seed):
    """Factory returning a callable that builds a fresh RodTrackingEnv.

    Wrapped this way because SubprocVecEnv needs *picklable callables*
    that build envs in worker processes, not instantiated env objects.
    """
    def _init():
        env = RodTrackingEnv(**{**ENV_BASE, 'young_modulus': E, 'p_static': p_static})
        env.reset(seed=seed)
        return env
    return _init


In [ ]:
# DAGGER DATA COLLECTION

def collect_dagger_data_parallel(
        student: nn.Module,
        experts: dict,
        device: torch.device,
        beta: float,
        round_num: int,
        episode_configs: list,
        n_parallel_envs: int = 8,
):
    """Roll out the student in parallel envs and collect expert labels.

    `beta` controls the mix: each parallel env independently follows
    the expert with probability `beta`, otherwise follows the
    student. Beta is typically a small warm-start value for round 1
    (e.g. 0.1) and 0 for subsequent rounds (pure on-policy data).

    The stored (state, action_label) pairs always use the EXPERT's
    label, regardless of who actually drives the env — that's the
    DAgger trick.
    """
    all_obs, all_logit, all_action = [], [], []

    for (E_phys, E_expert, n_dyn, n_stat) in episode_configs:
        expert = experts[E_expert]
        E_norm = normalize_E(E_phys)

        for regime, n_episodes_target, p_static in [
            ("dyn",  n_dyn,  0.0),
            ("stat", n_stat, 1.0),
        ]:
            # Per-env reproducible seeds.
            base_seed = round_num * 100_000 + int(E_phys / 1e5) * 100
            envs = SubprocVecEnv(
                [make_env(E_phys, p_static, base_seed + i * 7)
                 for i in range(n_parallel_envs)],
                start_method="spawn",
            )

            episodes_done = 0
            obs           = envs.reset()
            E_norm_col    = np.full((n_parallel_envs, 1), E_norm,
                                    dtype=np.float32)

            label_tag = (f"(label by E_exp={E_expert:.1e})"
                         if E_phys != E_expert else "")
            pbar = tqdm(total=n_episodes_target,
                        desc=f"R{round_num} {regime} "
                             f"E={E_phys:.1e} {label_tag}".strip())

            while episodes_done < n_episodes_target:
                # Expert labels the current batch of obs (raw, no E_norm).
                expert_actions, expert_logits = (
                    expert.get_actions_and_logits_batch(obs.astype(np.float32))
                )

                # Student predicts on the augmented obs (with E_norm).
                obs_aug   = np.concatenate([obs, E_norm_col], axis=1).astype(np.float32)
                obs_aug_t = torch.from_numpy(obs_aug).to(device)
                with torch.no_grad():
                    student_actions = torch.tanh(student(obs_aug_t)).cpu().numpy()

                # Mix student / expert actions per env.
                if beta > 0:
                    use_expert  = np.random.random(n_parallel_envs) < beta
                    env_actions = np.where(use_expert[:, None],
                                           expert_actions, student_actions)
                else:
                    env_actions = student_actions

                # Store: state visited by the student, action label from the expert.
                for i in range(n_parallel_envs):
                    all_obs.append(obs_aug[i].copy())
                    all_logit.append(expert_logits[i].copy())
                    all_action.append(expert_actions[i].copy())

                # Step the envs with the (mixed) actions.
                obs, _, dones, _ = envs.step(env_actions)

                # Count completed episodes.
                n_done = int(np.sum(dones))
                if n_done > 0:
                    pbar.update(min(n_done, n_episodes_target - episodes_done))
                    episodes_done += n_done

            pbar.close()
            envs.close()

    X = np.array(all_obs,    dtype=np.float32)
    y_logit  = np.array(all_logit,  dtype=np.float32)
    y_action = np.array(all_action, dtype=np.float32)
    print(f"  Round {round_num}: collected {len(X):,} new samples")
    return X, y_action, y_logit


In [ ]:
# Dataset size limiter

def limit_dataset(X_agg, y_logit_agg, y_action_agg, X_new, max_size):
    """Cap aggregated dataset at `max_size`, keeping all new samples.

    Parameters
    ----------
    X_agg, y_logit_agg, y_action_agg : aggregated arrays
                                       (new samples appended at the end)
    n_new   : how many of the trailing rows are the newly collected
              samples to be preserved
    max_size : cap; if `len(X_agg) <= max_size`, no-op
    """
    n_new = len(X_new)
    if len(X_agg) <= max_size:
        return X_agg, y_logit_agg, y_action_agg

    n_keep_old = max_size - n_new
    n_old = len(X_agg) - n_new
    old_indices = np.random.choice(n_old, n_keep_old, replace=False)
    new_indices = np.arange(n_old, len(X_agg))
    keep = np.concatenate([old_indices, new_indices])
    keep.sort()

    print(f"  Dataset limitato: {len(X_agg):,} → {max_size:,} "
          f"({n_keep_old:,} old + {n_new:,} new)")
    return X_agg[keep], y_logit_agg[keep], y_action_agg[keep]

In [ ]:
# TRAINING

def train_one_round(X_full, y_logit_full, student_path, round_num, out_dir):
    """Fine-tune the student on (X, y_logit) for one DAgger round.

    Returns the path to the best checkpoint (by val loss).
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    obs_dim = X_full.shape[1] - 1
    action_dim = y_logit_full.shape[1]

    # Start from the previous round's weights.
    model, _, _ = load_student(student_path, device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS_PER_ROUND, eta_min=1e-6)

    X_t = torch.tensor(X_full, dtype=torch.float32)
    y_t = torch.tensor(y_logit_full, dtype=torch.float32)

    n_val = int(len(X_t) * 0.15)
    n_train = len(X_t) - n_val

    dataset = TensorDataset(X_t, y_t)
    train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                    generator=torch.Generator().manual_seed(42))

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              pin_memory=True, num_workers=NUM_WORKERS,
                              persistent_workers=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            pin_memory=True, num_workers=NUM_WORKERS,
                            persistent_workers=True)

    best_val = float('inf')
    best_ep = 0
    early_stop = 0
    save_path = out_dir / f"student_dagger_r{round_num}.pth"

    print(f"\n  Training round {round_num}: {n_train:,} train, "
          f"{n_val:,} val, batch={BATCH_SIZE}, LR={LEARNING_RATE}")

    for epoch in range(EPOCHS_PER_ROUND):
        model.train()
        train_loss = 0.0
        for bx, by in train_loader:
            bx, by = bx.to(device, non_blocking=True), by.to(device, non_blocking=True)

            obs_noise = torch.randn(bx.shape[0], obs_dim, device=device) * OBS_NOISE_STD
            E_noise = torch.randn(bx.shape[0], 1, device=device) * E_NORM_NOISE_STD
            noisy_E = torch.clamp(bx[:, obs_dim:obs_dim+1] + E_noise, 0.0, 1.0)
            bx_noisy = torch.cat([bx[:, :obs_dim] + obs_noise, noisy_E], dim=1)

            optimizer.zero_grad()
            pred = model(bx_noisy)
            loss = criterion(pred, by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            train_loss += loss.item()

        avg_train = train_loss / len(train_loader)
        scheduler.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for bx, by in val_loader:
                bx, by = bx.to(device, non_blocking=True), by.to(device, non_blocking=True)
                val_loss += criterion(model(bx), by).item()
        avg_val = val_loss / len(val_loader)

        if avg_val < best_val:
            best_val = avg_val
            best_ep = epoch + 1
            early_stop = 0
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'val_loss': best_val,
                'obs_dim': obs_dim, 'action_dim': action_dim,
                'loss_type': 'logit_mse', 'dagger_round': round_num,
            }, save_path)
        else:
            early_stop += 1

        if (epoch + 1) % 5 == 0:
            lr = optimizer.param_groups[0]['lr']
            print(f"    Ep {epoch+1:>3d}/{EPOCHS_PER_ROUND} | "
                  f"Train: {avg_train:.5f} | Val: {avg_val:.5f} | LR: {lr:.1e}")

        if early_stop >= EARLY_STOP_PAT:
            print(f"    Early stop ep {epoch+1} (best={best_val:.5f} @ ep {best_ep})")
            break

    print(f"  Round {round_num} best: ep {best_ep}, val={best_val:.5f}")
    return str(save_path)

In [ ]:
# QUICK VALIDATION

def quick_validate(student_path, device, n_episodes=10):
    """Evaluate the student at a list of E values; return per-E metrics."""
    model, _, _ = load_student(student_path, device)
    results = {}
    for E in E_VALUES:
        E_norm = normalize_E(E)
        env = RodTrackingEnv(**{**ENV_BASE, 'young_modulus': E, 'p_static': 0.0})

        all_errors = []
        ep_on_goal = []

        for ep in range(n_episodes):
            obs, _ = env.reset(seed=ep * 7919)
            done = False
            ep_err = []
            while not done:
                obs_aug = np.concatenate([obs, [E_norm]]).astype(np.float32)
                with torch.no_grad():
                    x = torch.from_numpy(obs_aug).unsqueeze(0).to(device)
                    action = torch.tanh(model(x)).squeeze(0).cpu().numpy()
                obs, _, term, trunc, info = env.step(action)
                done = term or trunc
                ep_err.append(info["error"])
            all_errors.extend(ep_err)
            ep_on_goal.append(info["on_goal_fraction"])

        env.close()
        s1 = float(np.mean(np.array(all_errors) < 0.01)) * 100
        s2 = float(np.mean(np.array(all_errors) < 0.02)) * 100
        og = float(np.mean(ep_on_goal))
        results[E] = {"@1cm": s1, "@2cm": s2, "OnGoal": og}
    return results

def print_quick_validation(results: dict, title: str = ""):
    """Print the dict returned by `quick_validate`."""
    if title:
        print(f"\n  {title}")
    print(f"   {'E':<12} {'@1cm':>8} {'@2cm':>8} {'OnGoal':>8}")
    for E, r in results.items():
        print(f"   {E:<12.1e} {r['@1cm']:>7.1f}% "
              f"{r['@2cm']:>7.1f}% {r['OnGoal']:>7.3f}")


In [1]:
def main_dagger():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    print("\nLoading original distillation dataset")
    orig = np.load(ORIGINAL_DATASET)
    X_agg = orig['X'].copy()
    y_logit_agg = orig['y_logit'].copy()
    y_action_agg = orig.get('y_action', orig.get('y')).copy()
    print(f"   Original dataset: {X_agg.shape[0]:,} samples")

    print("\nLoading experts")
    experts = {}
    for E, config in EXPERTS.items():
        experts[E] = ExpertWrapper(config["model"], config["vecnorm"], E)
        print(f"   E={E:.0e} loaded")

    current_student_path = INITIAL_STUDENT

    print("\nBaseline validation")
    baseline = quick_validate(current_student_path, device)
    for E, r in baseline.items():
        print(f"   E={E:<10.0e} @1cm={r['@1cm']:>5.1f}% @2cm={r['@2cm']:>5.1f}% "
              f"OnGoal={r['OnGoal']:.3f}")

    all_results = {"baseline": baseline}

    for round_num in range(1, N_DAGGER_ROUNDS + 1):
        beta = BETA_SCHEDULE[min(round_num - 1, len(BETA_SCHEDULE) - 1)]

        print(f"\n{'='*60}")
        print(f"  DAGGER ROUND {round_num}/{N_DAGGER_ROUNDS}  (β={beta:.3f})")
        print(f"{'='*60}")

        student, _, _ = load_student(current_student_path, device)

        X_new, y_a_new, y_l_new = collect_dagger_data_parallel(
            student, experts, device, beta, round_num
        )

        X_agg = np.concatenate([X_agg, X_new], axis=0)
        y_logit_agg = np.concatenate([y_logit_agg, y_l_new], axis=0)
        y_action_agg = np.concatenate([y_action_agg, y_a_new], axis=0)

        # Limita dataset
        X_agg, y_logit_agg, y_action_agg = limit_dataset(
            X_agg, y_logit_agg, y_action_agg, X_new, MAX_DATASET_SIZE
        )
        print(f"  Current dataset: {X_agg.shape[0]:,} samples")

        np.savez(
            OUTPUT_DIR / f"dataset_dagger_r{round_num}.npz",
            X=X_agg, y_action=y_action_agg, y_logit=y_logit_agg,
        )

        round_dir = OUTPUT_DIR / f"round_{round_num}"
        round_dir.mkdir(exist_ok=True)
        current_student_path = train_one_round(
            X_agg, y_logit_agg, current_student_path, round_num, round_dir
        )

        results = quick_validate(current_student_path, device)
        print_quick_validation(results, title=f"Round {round_num} validation:")
        all_results[f"round_{round_num}"] = results

    final_path = OUTPUT_DIR / "student_dagger_final.pth"
    shutil.copy(current_student_path, final_path)
    print(f"\nFinal model:  {final_path}")

    print(f"\n{'='*70}")
    print(f"  STAGE 1 SUMMARY — @1cm by round")
    print(f"{'='*70}")
    print(f"{'Round':<12} ", end="")
    for E in E_VALUES:
        print(f"E={E:.0e} @1cm  ", end="")
    print()
    for label, res in all_results.items():
        print(f"{label:<12} ", end="")
        for E in E_VALUES:
            print(f"    {res[E]['@1cm']:>5.1f}%    ", end="")
        print()

    with open(OUTPUT_DIR / "dagger_summary.json", "w") as f:
        json.dump(all_results, f, indent=2, default=float)
    print(f"\nSummary saved to {OUTPUT_DIR}/")


main_dagger()

Device: cuda

1. Caricamento dataset originale...
   Dataset originale: 5,713,600 samples

2. Caricamento esperti...
   E=5e+06 caricato
   E=8e+06 caricato
   E=1e+07 caricato
   E=2e+07 caricato

3. Validazione baseline...
   E=5e+06      @1cm= 37.1% @2cm= 83.4% OnGoal=0.371
   E=8e+06      @1cm= 37.9% @2cm= 84.4% OnGoal=0.379
   E=1e+07      @1cm= 59.9% @2cm= 92.6% OnGoal=0.599
   E=2e+07      @1cm= 74.8% @2cm= 95.0% OnGoal=0.748

  DAGGER ROUND 1/3  (β=0.100)


R1 stat E=2.0e+07 β=0.100: 100%|██████████| 20/20 [01:17<00:00,  3.87s/it]


  Round 1: raccolti 1,828,352 nuovi sample
  Dataset limitato: 7,541,952 → 6,000,000 (4,171,648 old + 1,828,352 new)
  Dataset attuale: 6,000,000 samples

  Training round 1: 5,100,000 train, 900,000 val, batch=1024, LR=0.0002
    Ep   5/40 | Train: 0.25086 | Val: 0.20786 | LR: 1.9e-04
    Ep  10/40 | Train: 0.23764 | Val: 0.19498 | LR: 1.7e-04
    Ep  15/40 | Train: 0.23043 | Val: 0.18877 | LR: 1.4e-04
    Ep  20/40 | Train: 0.22565 | Val: 0.18518 | LR: 1.0e-04
    Ep  25/40 | Train: 0.22232 | Val: 0.18117 | LR: 6.2e-05
    Ep  30/40 | Train: 0.21999 | Val: 0.17908 | LR: 3.0e-05
    Ep  35/40 | Train: 0.21860 | Val: 0.17788 | LR: 8.6e-06
    Ep  40/40 | Train: 0.21813 | Val: 0.17747 | LR: 1.0e-06
  Round 1 best: ep 40, val=0.17747

  Validazione round 1...
   E=5e+06      @1cm= 55.8% @2cm= 93.4% OnGoal=0.558
   E=8e+06      @1cm= 54.2% @2cm= 90.7% OnGoal=0.542
   E=1e+07      @1cm= 72.6% @2cm= 96.4% OnGoal=0.725
   E=2e+07      @1cm= 86.3% @2cm= 96.8% OnGoal=0.863

  DAGGER ROUND 2/3 

R2 stat E=2.0e+07 β=0.000: 100%|██████████| 20/20 [00:41<00:00,  2.08s/it]


  Round 2: raccolti 1,828,352 nuovi sample
  Dataset limitato: 7,828,352 → 6,000,000 (4,171,648 old + 1,828,352 new)
  Dataset attuale: 6,000,000 samples

  Training round 2: 5,100,000 train, 900,000 val, batch=1024, LR=0.0002
    Ep   5/40 | Train: 0.27736 | Val: 0.23031 | LR: 1.9e-04
    Ep  10/40 | Train: 0.26401 | Val: 0.21979 | LR: 1.7e-04
    Ep  15/40 | Train: 0.25629 | Val: 0.21191 | LR: 1.4e-04
    Ep  20/40 | Train: 0.25103 | Val: 0.20713 | LR: 1.0e-04
    Ep  25/40 | Train: 0.24714 | Val: 0.20329 | LR: 6.2e-05
    Ep  30/40 | Train: 0.24451 | Val: 0.20076 | LR: 3.0e-05
    Ep  35/40 | Train: 0.24307 | Val: 0.19913 | LR: 8.6e-06
    Ep  40/40 | Train: 0.24246 | Val: 0.19877 | LR: 1.0e-06
  Round 2 best: ep 39, val=0.19877

  Validazione round 2...
   E=5e+06      @1cm= 58.8% @2cm= 93.6% OnGoal=0.588
   E=8e+06      @1cm= 58.9% @2cm= 94.1% OnGoal=0.588
   E=1e+07      @1cm= 78.2% @2cm= 96.4% OnGoal=0.782
   E=2e+07      @1cm= 90.6% @2cm= 97.2% OnGoal=0.906

  DAGGER ROUND 3/3 

R3 stat E=2.0e+07 β=0.000: 100%|██████████| 20/20 [00:48<00:00,  2.42s/it]


  Round 3: raccolti 1,828,352 nuovi sample
  Dataset limitato: 7,828,352 → 6,000,000 (4,171,648 old + 1,828,352 new)
  Dataset attuale: 6,000,000 samples

  Training round 3: 5,100,000 train, 900,000 val, batch=1024, LR=0.0002
    Ep   5/40 | Train: 0.27940 | Val: 0.23224 | LR: 1.9e-04
    Ep  10/40 | Train: 0.26787 | Val: 0.22180 | LR: 1.7e-04
    Ep  15/40 | Train: 0.26080 | Val: 0.21444 | LR: 1.4e-04
    Ep  20/40 | Train: 0.25584 | Val: 0.21115 | LR: 1.0e-04
    Ep  25/40 | Train: 0.25215 | Val: 0.20749 | LR: 6.2e-05
    Ep  30/40 | Train: 0.24962 | Val: 0.20470 | LR: 3.0e-05
    Ep  35/40 | Train: 0.24812 | Val: 0.20332 | LR: 8.6e-06
    Ep  40/40 | Train: 0.24755 | Val: 0.20284 | LR: 1.0e-06
  Round 3 best: ep 40, val=0.20284

  Validazione round 3...
   E=5e+06      @1cm= 60.5% @2cm= 94.3% OnGoal=0.605
   E=8e+06      @1cm= 61.7% @2cm= 94.1% OnGoal=0.617
   E=1e+07      @1cm= 79.8% @2cm= 97.0% OnGoal=0.798
   E=2e+07      @1cm= 90.1% @2cm= 97.3% OnGoal=0.901

✅ Modello finale: r

## Targeted DAgger

Adds 2 rounds of DAgger on **interpolated E values** to force smooth
behavior between the 4 trained experts. Labels for interpolated
points come from the *nearest* trained expert (e.g. 6.5e6 -> labeled
by 5e6 expert; 9e6 -> labeled by 1e7 expert).

Starts from the Stage 1 final student.

In [ ]:
# CONFIGURATION TARGETED

INITIAL_TARGETED = "results_dagger_fast/student_dagger_final.pth"

OUTPUT_DIR_TARGETED = Path("results_dagger_targeted")

TARGETED_E_VALUES = [
    (5e6,    5e6),       # trained expert
    (6.5e6,  5e6),       # interpolation -> labeled by 5e6
    (7.5e6,  7.5e6),     # trained expert
    (9e6,    1e7),       # interpolation -> labeled by 1e7
    (1e7,    1e7),       # trained expert
    (1.5e7,  2e7),       ## interpolation -> labeled by 2e7
    (2e7,    2e7),       # trained expert
]

# DAgger params
N_TARGETED_ROUNDS = 2
EPISODES_DYN_T    = 25
EPISODES_STAT_T   = 10
BETA_TARGETED     = 0.0
N_PARALLEL_T      = 8

# Training
EPOCHS_TARGETED   = 30
BATCH_SIZE_T      = 1024
LR_TARGETED       = 1e-4
EARLY_STOP_T      = 8
GRAD_CLIP_T       = 1.0
NUM_WORKERS_T     = 4

E_VALUES_VAL = [5e6, 6.5e6, 7.5e6, 9e6, 1e7, 1.5e7, 2e7]

In [2]:
# ======================================================================
# COLLECTION (versione targeted: usa E_phys per env, E_norm per student,
# expert labeling con esperto più vicino)
# ======================================================================

def collect_targeted_data(student, experts, device, beta, round_num):
    """
    Per ogni (E_phys, E_expert):
      - Crea env a young_modulus = E_phys
      - Lo student riceve E_norm corrispondente a E_phys
      - L'esperto E_expert etichetta gli stati visitati
    """
    all_obs, all_logit, all_action = [], [], []

    for (E_phys, E_expert) in TARGETED_E_VALUES:
        expert = experts[E_expert]
        E_norm = normalize_E(E_phys)

        for regime, n_ep, p_static in [
            ("dyn",  EPISODES_DYN_T,  0.0),
            ("stat", EPISODES_STAT_T, 1.0),
        ]:
            # Crea env paralleli con E_phys
            seed_base = round_num * 100000 + int(E_phys / 1e5)
            envs = SubprocVecEnv([
                make_env(E_phys, p_static, seed_base + i * 7)
                for i in range(N_PARALLEL_T)
            ], start_method="spawn")

            target_episodes = n_ep
            episodes_done = 0

            obs = envs.reset()
            pbar = tqdm(total=target_episodes,
                        desc=f"R{round_num} {regime} E_phys={E_phys:.1e} "
                             f"E_exp={E_expert:.0e}")

            E_norm_col = np.full((N_PARALLEL_T, 1), E_norm, dtype=np.float32)

            while episodes_done < target_episodes:
                # Esperto etichetta (usa il modello dell'esperto vicino)
                expert_actions, expert_logits = expert.get_actions_and_logits_batch(
                    obs.astype(np.float32)
                )

                # Student predice in batch
                obs_aug = np.concatenate([obs, E_norm_col], axis=1).astype(np.float32)
                obs_aug_t = torch.from_numpy(obs_aug).to(device)
                with torch.no_grad():
                    student_actions = torch.tanh(student(obs_aug_t)).cpu().numpy()

                # Mix (beta=0 → solo student)
                if beta > 0:
                    use_expert = np.random.random(N_PARALLEL_T) < beta
                    env_actions = np.where(
                        use_expert[:, None], expert_actions, student_actions
                    )
                else:
                    env_actions = student_actions

                # Salva: (obs+E_norm_intermedio, expert_logit_dell_esperto_vicino)
                for i in range(N_PARALLEL_T):
                    all_obs.append(obs_aug[i].copy())
                    all_logit.append(expert_logits[i].copy())
                    all_action.append(expert_actions[i].copy())

                obs, _, dones, _ = envs.step(env_actions)

                n_done = int(np.sum(dones))
                if n_done > 0:
                    episodes_done += n_done
                    pbar.update(min(n_done, target_episodes - (episodes_done - n_done)))

            pbar.close()
            envs.close()

    X = np.array(all_obs, dtype=np.float32)
    y_logit = np.array(all_logit, dtype=np.float32)
    y_action = np.array(all_action, dtype=np.float32)
    print(f"  Round {round_num}: raccolti {len(X):,} nuovi sample")
    return X, y_action, y_logit


# ======================================================================
# TRAINING ROUND (con noise su E_norm per preservare interpolazione)
# ======================================================================

def train_targeted_round(X_full, y_logit_full, student_path, round_num, out_dir):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    obs_dim = X_full.shape[1] - 1
    action_dim = y_logit_full.shape[1]

    model, _, _ = load_student(student_path, device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=LR_TARGETED)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS_TARGETED, eta_min=1e-6)

    X_t = torch.tensor(X_full, dtype=torch.float32)
    y_t = torch.tensor(y_logit_full, dtype=torch.float32)

    n_val = int(len(X_t) * 0.15)
    n_train = len(X_t) - n_val

    dataset = TensorDataset(X_t, y_t)
    train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                    generator=torch.Generator().manual_seed(42))

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE_T, shuffle=True,
                              pin_memory=True, num_workers=NUM_WORKERS_T,
                              persistent_workers=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE_T, shuffle=False,
                            pin_memory=True, num_workers=NUM_WORKERS_T,
                            persistent_workers=True)

    best_val = float('inf')
    best_ep = 0
    early_stop = 0
    save_path = out_dir / f"student_targeted_r{round_num}.pth"

    print(f"\n  Training targeted round {round_num}: {n_train:,} train, "
          f"{n_val:,} val, batch={BATCH_SIZE_T}, LR={LR_TARGETED}")

    for epoch in range(EPOCHS_TARGETED):
        model.train()
        train_loss = 0.0
        for bx, by in train_loader:
            bx = bx.to(device, non_blocking=True)
            by = by.to(device, non_blocking=True)

            # Noise su obs e su E_norm (preserva capacità interpolazione)
            obs_noise = torch.randn(bx.shape[0], obs_dim, device=device) * OBS_NOISE_STD
            E_noise = torch.randn(bx.shape[0], 1, device=device) * E_NORM_NOISE_STD
            noisy_E = torch.clamp(bx[:, obs_dim:obs_dim+1] + E_noise, 0.0, 1.0)
            bx_noisy = torch.cat([bx[:, :obs_dim] + obs_noise, noisy_E], dim=1)

            optimizer.zero_grad()
            pred = model(bx_noisy)
            loss = criterion(pred, by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_T)
            optimizer.step()
            train_loss += loss.item()

        avg_train = train_loss / len(train_loader)
        scheduler.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for bx, by in val_loader:
                bx = bx.to(device, non_blocking=True)
                by = by.to(device, non_blocking=True)
                val_loss += criterion(model(bx), by).item()
        avg_val = val_loss / len(val_loader)

        if avg_val < best_val:
            best_val = avg_val
            best_ep = epoch + 1
            early_stop = 0
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'val_loss': best_val,
                'obs_dim': obs_dim, 'action_dim': action_dim,
                'loss_type': 'logit_mse', 'targeted_round': round_num,
            }, save_path)
        else:
            early_stop += 1

        if (epoch + 1) % 5 == 0:
            lr = optimizer.param_groups[0]['lr']
            print(f"    Ep {epoch+1:>3d}/{EPOCHS_TARGETED} | "
                  f"Train: {avg_train:.5f} | Val: {avg_val:.5f} | LR: {lr:.1e}")

        if early_stop >= EARLY_STOP_T:
            print(f"    Early stop ep {epoch+1} (best={best_val:.5f} @ ep {best_ep})")
            break

    print(f"  Round {round_num} best: ep {best_ep}, val={best_val:.5f}")
    return str(save_path)


# ======================================================================
# QUICK VALIDATION (su tutti gli E inclusi interpolazioni)
# ======================================================================

def quick_validate_targeted(student_path, device, n_episodes=10):
    model, _, _ = load_student(student_path, device)
    results = {}
    for E in E_VALUES_VAL:
        E_norm = normalize_E(E)
        env = RodTrackingEnv(**{**ENV_BASE, 'young_modulus': E, 'p_static': 0.0})

        all_errors = []
        ep_on_goal = []
        for ep in range(n_episodes):
            obs, _ = env.reset(seed=ep * 7919)
            done = False
            ep_err = []
            while not done:
                obs_aug = np.concatenate([obs, [E_norm]]).astype(np.float32)
                with torch.no_grad():
                    x = torch.from_numpy(obs_aug).unsqueeze(0).to(device)
                    action = torch.tanh(model(x)).squeeze(0).cpu().numpy()
                obs, _, term, trunc, info = env.step(action)
                done = term or trunc
                ep_err.append(info["error"])
            all_errors.extend(ep_err)
            ep_on_goal.append(info["on_goal_fraction"])
        env.close()

        s1 = float(np.mean(np.array(all_errors) < 0.01)) * 100
        s2 = float(np.mean(np.array(all_errors) < 0.02)) * 100
        og = float(np.mean(ep_on_goal))
        results[E] = {"@1cm": s1, "@2cm": s2, "OnGoal": og}
    return results


# ======================================================================
# MAIN TARGETED
# ======================================================================

def main_targeted():
    OUTPUT_DIR_TARGETED.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    print(f"Modello iniziale: {INITIAL_TARGETED}")
    print(f"Output: {OUTPUT_DIR_TARGETED}")

    # 1. Carica dataset DAgger esistente come base
    print("\n1. Caricamento dataset DAgger esistente...")
    # Cerca l'ultimo dataset dagger salvato
    dagger_dataset_paths = sorted(Path("results_dagger_fast").glob("dataset_dagger_r*.npz"))
    if dagger_dataset_paths:
        ds_path = dagger_dataset_paths[-1]
        print(f"   Usando: {ds_path}")
        orig = np.load(ds_path)
    else:
        print("   Fallback su dataset originale")
        orig = np.load("distillation_data/distillation_dataset.npz")

    X_agg = orig['X'].copy()
    y_logit_agg = orig['y_logit'].copy()
    y_action_agg = orig.get('y_action', orig.get('y')).copy()
    print(f"   Dataset base: {X_agg.shape[0]:,} samples")

    # 2. Carica esperti (gli stessi del DAgger originale)
    print("\n2. Caricamento esperti...")
    experts = {}
    for E, config in EXPERTS.items():
        experts[E] = ExpertWrapper(config["model"], config["vecnorm"], E)
        print(f"   E={E:.0e} caricato")

    current_path = INITIAL_TARGETED

    # 3. Validazione baseline
    print("\n3. Validazione baseline (modello DAgger attuale)...")
    baseline = quick_validate_targeted(current_path, device, n_episodes=10)
    print(f"   {'E':<12} {'@1cm':>8} {'@2cm':>8} {'OnGoal':>8}")
    for E, r in baseline.items():
        print(f"   {E:<12.1e} {r['@1cm']:>7.1f}% {r['@2cm']:>7.1f}% {r['OnGoal']:>7.3f}")

    all_results = {"baseline": baseline}

    # 4. Round targeted
    for round_num in range(1, N_TARGETED_ROUNDS + 1):
        print(f"\n{'='*60}")
        print(f"  TARGETED ROUND {round_num}/{N_TARGETED_ROUNDS}  (β={BETA_TARGETED})")
        print(f"{'='*60}")

        student, _, _ = load_student(current_path, device)
        X_new, y_a_new, y_l_new = collect_targeted_data(
            student, experts, device, BETA_TARGETED, round_num
        )

        # Aggrega
        X_agg = np.concatenate([X_agg, X_new], axis=0)
        y_logit_agg = np.concatenate([y_logit_agg, y_l_new], axis=0)
        y_action_agg = np.concatenate([y_action_agg, y_a_new], axis=0)

        # Limita dataset (riusa MAX_DATASET_SIZE già definito)
        if len(X_agg) > MAX_DATASET_SIZE:
            n_new = len(X_new)
            n_keep_old = MAX_DATASET_SIZE - n_new
            old_idx = np.random.choice(len(X_agg) - n_new, n_keep_old, replace=False)
            new_idx = np.arange(len(X_agg) - n_new, len(X_agg))
            keep = np.concatenate([old_idx, new_idx])
            keep.sort()
            X_agg = X_agg[keep]
            y_logit_agg = y_logit_agg[keep]
            y_action_agg = y_action_agg[keep]
            print(f"  Dataset limitato a {MAX_DATASET_SIZE:,}")
        print(f"  Dataset attuale: {X_agg.shape[0]:,} samples")

        # Salva dataset
        np.savez(
            OUTPUT_DIR_TARGETED / f"dataset_targeted_r{round_num}.npz",
            X=X_agg, y_action=y_action_agg, y_logit=y_logit_agg,
        )

        # Training
        round_dir = OUTPUT_DIR_TARGETED / f"round_{round_num}"
        round_dir.mkdir(exist_ok=True)
        current_path = train_targeted_round(
            X_agg, y_logit_agg, current_path, round_num, round_dir
        )

        # Validazione
        print(f"\n  Validazione round {round_num}...")
        results = quick_validate_targeted(current_path, device, n_episodes=10)
        print(f"   {'E':<12} {'@1cm':>8} {'@2cm':>8} {'OnGoal':>8}")
        for E, r in results.items():
            print(f"   {E:<12.1e} {r['@1cm']:>7.1f}% {r['@2cm']:>7.1f}% {r['OnGoal']:>7.3f}")
        all_results[f"round_{round_num}"] = results

    # 5. Salva modello finale
    import shutil
    final_path = OUTPUT_DIR_TARGETED / "student_targeted_final.pth"
    shutil.copy(current_path, final_path)
    print(f"\n✅ Modello finale: {final_path}")

    # 6. Riepilogo
    print(f"\n{'='*70}")
    print(f"  RIEPILOGO TARGETED DAGGER")
    print(f"{'='*70}")
    print(f"{'Round':<12}", end="")
    for E in E_VALUES_VAL:
        print(f"  E={E:.1e}", end="")
    print()
    print("-" * 80)
    for label, res in all_results.items():
        print(f"{label:<12}", end="")
        for E in E_VALUES_VAL:
            print(f"   {res[E]['@1cm']:>5.1f}%", end="")
        print()

    import json
    with open(OUTPUT_DIR_TARGETED / "targeted_summary.json", "w") as f:
        json.dump(all_results, f, indent=2, default=float)
    print(f"\n✅ Tutto salvato in {OUTPUT_DIR_TARGETED}/")


# ======================================================================
# ESEGUI
# ======================================================================
if __name__ == "__main__":
    main_targeted()

Device: cuda
Modello iniziale: results_dagger_fast/student_dagger_final.pth
Output: results_dagger_targeted

1. Caricamento dataset DAgger esistente...
   Usando: results_dagger_fast\dataset_dagger_r3.npz
   Dataset base: 6,000,000 samples

2. Caricamento esperti...
   E=5e+06 caricato
   E=8e+06 caricato
   E=1e+07 caricato
   E=2e+07 caricato

3. Validazione baseline (modello DAgger attuale)...
   E                @1cm     @2cm   OnGoal
   5.0e+06         60.5%    94.3%   0.605
   6.5e+06         54.8%    93.3%   0.548
   7.5e+06         61.7%    94.1%   0.617
   9.0e+06         75.7%    96.9%   0.757
   1.0e+07         79.8%    97.0%   0.798
   1.5e+07         88.7%    97.6%   0.887
   2.0e+07         90.1%    97.3%   0.901

  TARGETED ROUND 1/2  (β=0.0)


R1 stat E_phys=2.0e+07 E_exp=2e+07: 100%|██████████| 10/10 [00:29<00:00,  2.98s/it]


  Round 1: raccolti 2,399,712 nuovi sample
  Dataset limitato a 6,000,000
  Dataset attuale: 6,000,000 samples

  Training targeted round 1: 5,100,000 train, 900,000 val, batch=1024, LR=0.0001
    Ep   5/30 | Train: 0.33470 | Val: 0.26935 | LR: 9.3e-05
    Ep  10/30 | Train: 0.31891 | Val: 0.25494 | LR: 7.5e-05
    Ep  15/30 | Train: 0.31079 | Val: 0.24831 | LR: 5.1e-05
    Ep  20/30 | Train: 0.30609 | Val: 0.24407 | LR: 2.6e-05
    Ep  25/30 | Train: 0.30372 | Val: 0.24187 | LR: 7.6e-06
    Ep  30/30 | Train: 0.30301 | Val: 0.24127 | LR: 1.0e-06
  Round 1 best: ep 30, val=0.24127

  Validazione round 1...
   E                @1cm     @2cm   OnGoal
   5.0e+06         61.7%    92.7%   0.617
   6.5e+06         76.3%    97.7%   0.762
   7.5e+06         69.4%    95.7%   0.694
   9.0e+06         77.7%    96.4%   0.777
   1.0e+07         83.3%    97.2%   0.833
   1.5e+07         89.1%    97.6%   0.891
   2.0e+07         91.9%    97.3%   0.919

  TARGETED ROUND 2/2  (β=0.0)


R2 stat E_phys=2.0e+07 E_exp=2e+07: 100%|██████████| 10/10 [00:26<00:00,  2.69s/it]


  Round 2: raccolti 2,399,712 nuovi sample
  Dataset limitato a 6,000,000
  Dataset attuale: 6,000,000 samples

  Training targeted round 2: 5,100,000 train, 900,000 val, batch=1024, LR=0.0001
    Ep   5/30 | Train: 0.33527 | Val: 0.26667 | LR: 9.3e-05
    Ep  10/30 | Train: 0.32085 | Val: 0.25521 | LR: 7.5e-05
    Ep  15/30 | Train: 0.31302 | Val: 0.24804 | LR: 5.1e-05
    Ep  20/30 | Train: 0.30850 | Val: 0.24421 | LR: 2.6e-05
    Ep  25/30 | Train: 0.30607 | Val: 0.24197 | LR: 7.6e-06
    Ep  30/30 | Train: 0.30541 | Val: 0.24138 | LR: 1.0e-06
  Round 2 best: ep 30, val=0.24138

  Validazione round 2...
   E                @1cm     @2cm   OnGoal
   5.0e+06         61.7%    93.8%   0.617
   6.5e+06         73.9%    96.8%   0.739
   7.5e+06         71.3%    96.4%   0.713
   9.0e+06         79.0%    97.1%   0.790
   1.0e+07         81.4%    97.3%   0.814
   1.5e+07         89.2%    97.2%   0.892
   2.0e+07         93.0%    97.3%   0.929

✅ Modello finale: results_dagger_targeted\studen